# 03 · Preprocessing QA (Interview Notebook)

This notebook is for **assignment presentation** and validates that preprocessing is correct before training.

Checks included:
- merged dataset size and source mix
- extracted label distribution
- extraction quality report (failures)
- schema sanity checks
- sample records for manual review
- split preview (train/valid/test)

In [ ]:
import json
from pathlib import Path
import random

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
ROOT = Path().resolve().parent

MERGED_PATH = ROOT / 'data' / 'raw' / 'conversations_merged.json'
EXTRACTIONS_PATH = ROOT / 'data' / 'processed' / 'extractions.jsonl'
EXTRACTION_REPORT_PATH = ROOT / 'data' / 'processed' / 'extraction_report.json'
TRAINING_TABLE_PATH = ROOT / 'data' / 'processed' / 'training_table.csv'

print('Paths configured')

## 1) Raw merged dataset sanity

In [ ]:
with open(MERGED_PATH, encoding='utf-8') as f:
    merged = json.load(f)

print(f'Total merged conversations: {len(merged)}')
src_counts = pd.Series([r.get('source', 'unknown') for r in merged]).value_counts()
display(src_counts.to_frame('count'))

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
src_counts.plot(kind='bar', ax=ax, color=['#4c78a8','#f58518','#54a24b','#e45756'])
ax.set_title('Source composition')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

## 2) Extraction report QA

In [ ]:
with open(EXTRACTION_REPORT_PATH, encoding='utf-8') as f:
    rep = json.load(f)

stats = rep.get('stats', {})
display(pd.DataFrame([stats]))

total = stats.get('total', 0)
ok = stats.get('ok', 0)
print(f'Extraction success rate: {ok/max(total,1):.2%}')
print('Label counts:', rep.get('label_counts', {}))

In [ ]:
failures = rep.get('failure_examples', [])
if failures:
    fail_df = pd.DataFrame(failures)
    display(fail_df.head(10))
    if 'reason' in fail_df.columns:
        fig, ax = plt.subplots(figsize=(7,4))
        fail_df['reason'].value_counts().plot(kind='bar', ax=ax)
        ax.set_title('Top extraction failure reasons (sample)')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print('No failure examples recorded.')

## 3) Flattened training table QA

In [ ]:
df = pd.read_csv(TRAINING_TABLE_PATH)
print(df.shape)
display(df.head(3))

In [ ]:
label_cols = [c for c in df.columns if c.startswith('label_')]
feat_cols = [c for c in df.columns if c.startswith('feat_')]

print('label cols:', label_cols)
print('n_feat_cols:', len(feat_cols))
print('missing guest_text:', int((df['guest_text'].fillna('').str.strip() == '').sum()))

In [ ]:
label_counts = df[label_cols].sum().sort_values(ascending=False)
display(label_counts.to_frame('positives'))

fig, ax = plt.subplots(figsize=(8,4))
label_counts.plot(kind='bar', ax=ax, color='#4c78a8')
ax.set_title('Positive labels by category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

## 4) Manual review samples (important for interview)

In [ ]:
random.seed(42)
sample_idx = random.sample(range(len(df)), min(5, len(df)))
for i in sample_idx:
    row = df.iloc[i]
    print('='*80)
    print('record_id:', row.get('record_id', 'n/a'))
    print('labels:', {c: int(row[c]) for c in label_cols})
    print('text:', row['guest_text'][:500])
    print('='*80)

## 5) Train/valid/test split preview

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print('train:', train_df.shape)
print('valid:', valid_df.shape)
print('test :', test_df.shape)

split_summary = pd.DataFrame({
    'train_pos': train_df[label_cols].sum(),
    'valid_pos': valid_df[label_cols].sum(),
    'test_pos': test_df[label_cols].sum(),
})
display(split_summary)